In [19]:
import pandas as pd
import numpy as np

df_nav = pd.read_csv('../data/raw/02_nav_history.csv')
df_transactions = pd.read_csv('../data/raw/08_investor_transactions.csv')
df_performance = pd.read_csv('../data/raw/07_scheme_performance.csv')
df_fund_master = pd.read_csv('../data/raw/01_fund_master.csv')

print("nav_history:", df_nav.shape)
print("investor_transactions:", df_transactions.shape)
print("scheme_performance:", df_performance.shape)
print("fund_master:", df_fund_master.shape)

nav_history: (46000, 3)
investor_transactions: (32778, 13)
scheme_performance: (40, 19)
fund_master: (40, 15)


In [20]:
df_nav['date'] = pd.to_datetime(df_nav['date'])
print(df_nav.dtypes)

amfi_code             int64
date         datetime64[us]
nav                 float64
dtype: object


In [21]:
df_nav = df_nav.sort_values(by=['amfi_code', 'date']).reset_index(drop=True)
df_nav.head(10)

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639
5,100016,2022-01-10,510.7136
6,100016,2022-01-11,513.5542
7,100016,2022-01-12,512.3195
8,100016,2022-01-13,510.2445
9,100016,2022-01-14,514.3636


In [22]:
duplicate_count = df_nav.duplicated(subset=['amfi_code', 'date']).sum()
print("Duplicate (amfi_code, date) combinations:", duplicate_count)

Duplicate (amfi_code, date) combinations: 0


In [23]:
sample_fund = df_nav[df_nav['amfi_code'] == 100016].copy()
sample_fund['day_gap'] = sample_fund['date'].diff().dt.days

print("Gap sizes between consecutive NAV dates (sample fund):")
print(sample_fund['day_gap'].value_counts())

Gap sizes between consecutive NAV dates (sample fund):
day_gap
1.0    920
3.0    229
Name: count, dtype: int64


In [24]:
def fill_calendar_for_fund(amfi_code, group):
    full_range = pd.date_range(start=group['date'].min(), end=group['date'].max(), freq='D')
    group = group.set_index('date').reindex(full_range)
    group['nav'] = group['nav'].ffill()
    group['amfi_code'] = amfi_code
    group.index.name = 'date'
    return group.reset_index()

filled_pieces = []
for code, group in df_nav.groupby('amfi_code'):
    filled_pieces.append(fill_calendar_for_fund(code, group))

df_nav_filled = pd.concat(filled_pieces, ignore_index=True)

print("Original shape:", df_nav.shape)
print("After filling calendar gaps:", df_nav_filled.shape)
df_nav_filled.head(10)

Original shape: (46000, 3)
After filling calendar gaps: (64320, 3)


,date,amfi_code,nav
0,2022-01-03,100016,520.4608
1,2022-01-04,100016,515.0971
2,2022-01-05,100016,521.7239
3,2022-01-06,100016,515.7880
4,2022-01-07,100016,515.1639
5,2022-01-08,100016,515.1639
6,2022-01-09,100016,515.1639
7,2022-01-10,100016,510.7136
8,2022-01-11,100016,513.5542
9,2022-01-12,100016,512.3195


In [25]:
invalid_nav_count = (df_nav_filled['nav'] <= 0).sum()
print("Number of NAV values <= 0:", invalid_nav_count)

print("\nMinimum NAV value:", df_nav_filled['nav'].min())
print("Maximum NAV value:", df_nav_filled['nav'].max())

Number of NAV values <= 0: 0

Minimum NAV value: 26.1366
Maximum NAV value: 4268.5497


In [26]:
df_nav_filled.to_csv('../data/processed/02_nav_history_cleaned.csv', index=False)
print("Saved cleaned nav_history to data/processed/")

Saved cleaned nav_history to data/processed/


In [27]:
print("Unique transaction_type values:", df_transactions['transaction_type'].unique())
print("\nUnique kyc_status values:", df_transactions['kyc_status'].unique())
print("\nAmount range:", df_transactions['amount_inr'].min(), "to", df_transactions['amount_inr'].max())
print("\nTransaction_date sample:", df_transactions['transaction_date'].head(3).tolist())

Unique transaction_type values: <StringArray>
['SIP', 'Redemption', 'Lumpsum']
Length: 3, dtype: str

Unique kyc_status values: <StringArray>
['Verified', 'Pending']
Length: 2, dtype: str

Amount range: 400 to 597498

Transaction_date sample: ['2024-01-01', '2024-01-01', '2024-01-01']


In [28]:
invalid_amount_count = (df_transactions['amount_inr'] <= 0).sum()
print("Transactions with amount <= 0:", invalid_amount_count)

print("\nMissing values per column:\n", df_transactions.isna().sum())

Transactions with amount <= 0: 0

Missing values per column:
 investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64


In [29]:
df_transactions['transaction_date'] = pd.to_datetime(df_transactions['transaction_date'])
print(df_transactions.dtypes)

investor_id                      str
transaction_date      datetime64[us]
amfi_code                      int64
transaction_type                 str
amount_inr                     int64
state                            str
city                             str
city_tier                        str
age_group                        str
gender                           str
annual_income_lakh           float64
payment_mode                     str
kyc_status                       str
dtype: object


In [30]:
df_transactions.to_csv('../data/processed/08_investor_transactions_cleaned.csv', index=False)
print("Saved cleaned investor_transactions to data/processed/")

Saved cleaned investor_transactions to data/processed/


In [31]:
# Check expense ratio range
out_of_range = df_performance[(df_performance['expense_ratio_pct'] < 0.1) | (df_performance['expense_ratio_pct'] > 2.5)]
print("Funds with expense_ratio outside 0.1%-2.5%:", len(out_of_range))
if len(out_of_range) > 0:
    print(out_of_range[['scheme_name', 'expense_ratio_pct']])

# Check returns for negative/unrealistic values
print("\nReturn 1yr range:", df_performance['return_1yr_pct'].min(), "to", df_performance['return_1yr_pct'].max())
print("Return 3yr range:", df_performance['return_3yr_pct'].min(), "to", df_performance['return_3yr_pct'].max())
print("Return 5yr range:", df_performance['return_5yr_pct'].min(), "to", df_performance['return_5yr_pct'].max())

Funds with expense_ratio outside 0.1%-2.5%: 0

Return 1yr range: 4.26 to 24.93
Return 3yr range: 5.14 to 23.39
Return 5yr range: 5.43 to 23.8


In [32]:
df_aum = pd.read_csv('../data/raw/03_aum_by_fund_house.csv')
df_sip = pd.read_csv('../data/raw/04_monthly_sip_inflows.csv')
df_category = pd.read_csv('../data/raw/05_category_inflows.csv')
df_folio = pd.read_csv('../data/raw/06_industry_folio_count.csv')
df_holdings = pd.read_csv('../data/raw/09_portfolio_holdings.csv')
df_benchmark = pd.read_csv('../data/raw/10_benchmark_indices.csv')

print("All 6 remaining files loaded")
print("aum:", df_aum.shape, "| sip:", df_sip.shape, "| category:", df_category.shape)
print("folio:", df_folio.shape, "| holdings:", df_holdings.shape, "| benchmark:", df_benchmark.shape)

All 6 remaining files loaded
aum: (90, 5) | sip: (48, 6) | category: (144, 3)
folio: (21, 6) | holdings: (322, 8) | benchmark: (8050, 3)


In [33]:
df_fund_master['launch_date'] = pd.to_datetime(df_fund_master['launch_date'])
df_aum['date'] = pd.to_datetime(df_aum['date'])
df_holdings['portfolio_date'] = pd.to_datetime(df_holdings['portfolio_date'])
df_benchmark['date'] = pd.to_datetime(df_benchmark['date'])

# These two use "month" as quarterly/monthly periods, not exact dates - convert carefully
df_sip['month'] = pd.to_datetime(df_sip['month'], format='%Y-%m')
df_category['month'] = pd.to_datetime(df_category['month'], format='%Y-%m')
df_folio['month'] = pd.to_datetime(df_folio['month'], format='%Y-%m')

print("All date columns converted")

All date columns converted


In [34]:
df_fund_master.to_csv('../data/processed/01_fund_master_cleaned.csv', index=False)
df_aum.to_csv('../data/processed/03_aum_by_fund_house_cleaned.csv', index=False)
df_sip.to_csv('../data/processed/04_monthly_sip_inflows_cleaned.csv', index=False)
df_category.to_csv('../data/processed/05_category_inflows_cleaned.csv', index=False)
df_folio.to_csv('../data/processed/06_industry_folio_count_cleaned.csv', index=False)
df_holdings.to_csv('../data/processed/09_portfolio_holdings_cleaned.csv', index=False)
df_benchmark.to_csv('../data/processed/10_benchmark_indices_cleaned.csv', index=False)

print("All 7 remaining files saved to data/processed/")

All 7 remaining files saved to data/processed/


In [36]:
from sqlalchemy import create_engine, text

engine = create_engine('sqlite:///../dashboard/bluestock_mf.db')

with open('../sql/schema.sql', 'r') as f:
    schema_sql = f.read()

with engine.connect() as conn:
    for statement in schema_sql.split(';'):
        statement = statement.strip()
        if statement:
            conn.execute(text(statement))
    conn.commit()

print("Database created and schema applied successfully")

Database created and schema applied successfully


In [37]:
all_dates = pd.date_range(start='2022-01-01', end='2026-05-29', freq='D')

dim_date = pd.DataFrame({'date': all_dates})
dim_date['year'] = dim_date['date'].dt.year
dim_date['month'] = dim_date['date'].dt.month
dim_date['quarter'] = dim_date['date'].dt.quarter
dim_date['day_of_week'] = dim_date['date'].dt.day_name()
dim_date['is_weekend'] = dim_date['date'].dt.dayofweek.isin([5, 6]).astype(int)

dim_date['date'] = dim_date['date'].dt.strftime('%Y-%m-%d')

print(dim_date.shape)
dim_date.head()

(1610, 6)


,date,year,month,quarter,day_of_week,is_weekend
0,2022-01-01,2022,1,1,Saturday,1
1,2022-01-02,2022,1,1,Sunday,1
2,2022-01-03,2022,1,1,Monday,0
3,2022-01-04,2022,1,1,Tuesday,0
4,2022-01-05,2022,1,1,Wednesday,0


In [38]:
df_fund_master.to_sql('dim_fund', engine, if_exists='replace', index=False)
dim_date.to_sql('dim_date', engine, if_exists='replace', index=False)

print("dim_fund and dim_date loaded")

dim_fund and dim_date loaded


In [39]:
df_nav_filled.to_sql('fact_nav', engine, if_exists='replace', index=False)
df_transactions.to_sql('fact_transactions', engine, if_exists='replace', index=False)
df_performance.to_sql('fact_performance', engine, if_exists='replace', index=False)
df_aum.to_sql('fact_aum', engine, if_exists='replace', index=False)

print("All fact tables loaded")

All fact tables loaded


In [40]:
with engine.connect() as conn:
    for table in ['dim_fund', 'dim_date', 'fact_nav', 'fact_transactions', 'fact_performance', 'fact_aum']:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        count = result.scalar()
        print(f"{table}: {count} rows")

dim_fund: 40 rows
dim_date: 1610 rows
fact_nav: 64320 rows
fact_transactions: 32778 rows
fact_performance: 40 rows
fact_aum: 90 rows
